# Building Ministral-3 (8B) Multimodal from Scratch
This notebook implements the complete architecture for `mistralai/Ministral-3-8B-Instruct-2512`. 
We build the `Pixtral` vision encoder, the `Ministral` text decoder, the `Mistral3` multimodal projector, and an efficient autoregressive decoding loop utilizing a custom `KVCache`.

### Step 1: Imports and Environment Setup

In [ ]:
import os
import gc
from typing import List, Tuple, Optional, Dict, Any, Iterable
from dataclasses import dataclass, field
from pathlib import Path

import torch
import torch.nn as nn
from safetensors import safe_open
from huggingface_hub import snapshot_download, login
from transformers import AutoProcessor

# Configuration
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float32

# Optional: Log in to Hugging Face if using gated models
# login(token="hf_YOUR_TOKEN_HERE")

### Step 2: Utilities (KV Cache & RoPE Rotations)
To ensure fast text generation, we use a custom `KVCache` that stores attention keys and values. We also define the trigonometric rotation functions needed for Rotary Positional Embeddings (RoPE).

In [ ]:
class KVCache:
    """Key-Value cache for efficient transformer inference."""
    def __init__(self) -> None:
        self.key_cache: List[torch.Tensor] = []
        self.value_cache: List[torch.Tensor] = []
        
    def num_items(self) -> int:
        if not self.key_cache:
            return 0
        return self.key_cache[0].shape[2]

    def update(self, new_key: torch.Tensor, new_value: torch.Tensor, layer_idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        if len(self.key_cache) <= layer_idx:
            self.key_cache.append(new_key)
            self.value_cache.append(new_value)
        else:
            self.key_cache[layer_idx] = torch.cat([self.key_cache[layer_idx], new_key], dim=-2)
            self.value_cache[layer_idx] = torch.cat([self.value_cache[layer_idx], new_value], dim=-2)
        return self.key_cache[layer_idx], self.value_cache[layer_idx]

def rotate_half(hidden_state: torch.Tensor) -> torch.Tensor:
    x1 = hidden_state[..., : hidden_state.shape[-1] // 2]
    x2 = hidden_state[..., hidden_state.shape[-1] // 2 :]
    return torch.cat((-x2, x1), dim=-1)

def apply_rotary_pos_emb(q: torch.Tensor, k: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor, unsqueeze_dim: int = 1) -> tuple:
    cos = cos.unsqueeze(unsqueeze_dim)
    sin = sin.unsqueeze(unsqueeze_dim)
    q_embed = (q * cos) + (rotate_half(q) * sin)
    k_embed = (k * cos) + (rotate_half(k) * sin)
    return q_embed, k_embed

### Step 3: Vision Tower (Pixtral)
The vision encoder uses a 2D-Grid based RoPE and processes images into patches. We define its architecture layers here.
*(Note: We applied the `.unsqueeze(0)` fix to `position_ids` directly in the `PixtralVisionModel` forward pass).*

In [ ]:
@dataclass
class PixtralConfig:
    head_dim: int = 64
    num_heads: int = 16
    attention_dropout: float = 0.0
    hidden_size: int = 1024
    image_size: int = 1540
    intermediate_size: int = 4096
    num_attention_heads: int = 16
    num_hidden_layers: int = 24
    patch_size: int = 14
    rope_theta: float = 10000.0
    num_channels: int = 3

def position_ids_in_meshgrid(patch_embeds_list, max_width):
    positions = []
    for patch in patch_embeds_list:
        height, width = patch.shape[-2:]
        mesh = torch.meshgrid(torch.arange(height), torch.arange(width), indexing="ij")
        h_grid, v_grid = torch.stack(mesh, dim=-1).reshape(-1, 2).chunk(2, -1)
        ids = h_grid * max_width + v_grid
        positions.append(ids[:, 0])
    return torch.cat(positions)

def generate_block_attention_mask(patch_embeds_list, tensor):
    dtype, device = tensor.dtype, tensor.device
    seq_len = tensor.shape[1]
    causal_mask = torch.full((seq_len, seq_len), fill_value=torch.finfo(dtype).min, dtype=dtype, device=device)
    block_end_idx = torch.tensor(patch_embeds_list).cumsum(-1)
    block_start_idx = torch.tensor([0] + patch_embeds_list[:-1]).cumsum(-1)
    for start, end in zip(block_start_idx, block_end_idx):
        causal_mask[start:end, start:end] = 0
    return causal_mask[None, None, :, :].expand(tensor.shape[0], 1, -1, -1)

class PixtralRotaryEmbedding(nn.Module):
    def __init__(self, config: PixtralConfig) -> None:
        super().__init__()
        self.config = config
        inv_freq = self.compute_default_rope_parameters()
        self.register_buffer("inv_freq", inv_freq, persistent=False)

    def compute_default_rope_parameters(self) -> torch.Tensor:
        dim = self.config.head_dim
        max_patches_per_side = self.config.image_size // self.config.patch_size
        h, w = torch.arange(max_patches_per_side), torch.arange(max_patches_per_side)
        freqs = 1.0 / (self.config.rope_theta ** (torch.arange(0, dim, 2).float() / dim))
        freqs_h = torch.outer(h, freqs[::2]).float()
        freqs_w = torch.outer(w, freqs[1::2]).float()
        inv_freq = torch.cat([
            freqs_h[:, None, :].repeat(1, max_patches_per_side, 1),
            freqs_w[None, :, :].repeat(max_patches_per_side, 1, 1),
        ], dim=-1).reshape(-1, dim // 2)
        return torch.cat((inv_freq, inv_freq), dim=-1)

    def forward(self, hidden_state: torch.Tensor, position_ids: torch.Tensor) -> tuple:
        freqs = self.inv_freq[position_ids]
        return freqs.cos().to(hidden_state.dtype), freqs.sin().to(hidden_state.dtype)

class PixtralRMSNorm(nn.Module):
    def __init__(self, hidden_size: int, eps: float = 1e-6):
        super().__init__()
        self.variance_epsilon = eps
        self.weight = nn.Parameter(torch.ones(hidden_size))

    def forward(self, hidden_states: torch.Tensor) -> torch.Tensor:
        input_dtype = hidden_states.dtype
        hidden_states = hidden_states.to(torch.float32)
        variance = hidden_states.pow(2).mean(-1, keepdim=True)
        hidden_states = hidden_states * torch.rsqrt(variance + self.variance_epsilon)
        return self.weight * hidden_states.to(input_dtype)

class PixtralMLP(nn.Module):
    def __init__(self, config: PixtralConfig):
        super().__init__()
        self.gate_proj = nn.Linear(config.hidden_size, config.intermediate_size, bias=False)
        self.up_proj = nn.Linear(config.hidden_size, config.intermediate_size, bias=False)
        self.down_proj = nn.Linear(config.intermediate_size, config.hidden_size, bias=False)
        self.act_fn = nn.SiLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.down_proj(self.act_fn(self.up_proj(x) * self.gate_proj(x)))

class PixtralAttention(nn.Module):
    def __init__(self, config: PixtralConfig):
        super().__init__()
        self.hidden_size, self.num_attention_heads, self.head_dim = config.hidden_size, config.num_attention_heads, config.head_dim
        self.q_proj = nn.Linear(self.hidden_size, self.hidden_size, bias=False)
        self.k_proj = nn.Linear(self.hidden_size, self.hidden_size, bias=False)
        self.v_proj = nn.Linear(self.hidden_size, self.hidden_size, bias=False)
        self.o_proj = nn.Linear(self.hidden_size, self.hidden_size, bias=False)
        self.attn_dropout = nn.Dropout(config.attention_dropout)

    def forward(self, hidden_states: torch.Tensor, attention_mask: Optional[torch.Tensor] = None, position_embeddings: Optional[tuple] = None):
        batch_size, patches, _ = hidden_states.size()
        query = self.q_proj(hidden_states).view(batch_size, patches, self.num_attention_heads, self.head_dim).transpose(1, 2)
        key = self.k_proj(hidden_states).view(batch_size, patches, self.num_attention_heads, self.head_dim).transpose(1, 2)
        value = self.v_proj(hidden_states).view(batch_size, patches, self.num_attention_heads, self.head_dim).transpose(1, 2)

        if position_embeddings is not None:
            # FIX: Ensure unsqueeze_dim=1 for broadcasting
            query, key = apply_rotary_pos_emb(query, key, position_embeddings[0], position_embeddings[1], unsqueeze_dim=1)

        attn_scores = torch.matmul(query, key.transpose(-2, -1)) * (1.0 / (self.head_dim**0.5))
        if attention_mask is not None: attn_scores = attn_scores + attention_mask
        
        attn_weights = self.attn_dropout(torch.softmax(attn_scores, dim=-1))
        attn_output = torch.matmul(attn_weights, value).transpose(1, 2).contiguous().view(batch_size, patches, self.hidden_size)
        return self.o_proj(attn_output), None

class PixtralAttentionLayer(nn.Module):
    def __init__(self, config: PixtralConfig):
        super().__init__()
        self.attention_norm = PixtralRMSNorm(config.hidden_size)
        self.ffn_norm = PixtralRMSNorm(config.hidden_size)
        self.feed_forward = PixtralMLP(config)
        self.attention = PixtralAttention(config)

    def forward(self, hidden_states, attention_mask, position_embeddings):
        residual = hidden_states
        hidden_states = self.attention_norm(hidden_states)
        attn_out, _ = self.attention(hidden_states, attention_mask, position_embeddings)
        hidden_states = residual + attn_out
        
        residual = hidden_states
        hidden_states = self.feed_forward(self.ffn_norm(hidden_states))
        return (residual + hidden_states,)

class PixtralTransformer(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.layers = nn.ModuleList([PixtralAttentionLayer(config) for _ in range(config.num_hidden_layers)])

    def forward(self, hidden_states, attention_mask, position_embeddings, output_hidden_states=False):
        encoder_states = []
        for layer in self.layers:
            if output_hidden_states: encoder_states.append(hidden_states)
            hidden_states = layer(hidden_states, attention_mask, position_embeddings)[0]
        if output_hidden_states: encoder_states.append(hidden_states)
        
        return {"last_hidden_state": hidden_states, "hidden_states": encoder_states if output_hidden_states else None}

class PixtralVisionModel(nn.Module):
    def __init__(self, config: PixtralConfig):
        super().__init__()
        self.config = config
        self.patch_conv = nn.Conv2d(config.num_channels, config.hidden_size, kernel_size=config.patch_size, stride=config.patch_size, bias=False)
        self.patch_size = config.patch_size
        self.ln_pre = PixtralRMSNorm(config.hidden_size, eps=1e-5)
        self.transformer = PixtralTransformer(config)
        self.patch_positional_embedding = PixtralRotaryEmbedding(config)

    def forward(self, pixel_values: torch.Tensor, image_sizes: torch.Tensor | None = None, output_hidden_states: bool = False):
        if image_sizes is None:
            image_sizes = torch.tensor([(pixel_values.shape[2], pixel_values.shape[3])] * pixel_values.shape[0])

        patch_embeds = self.patch_conv(pixel_values.to(dtype=self.patch_conv.weight.dtype))
        patch_embeds_list = [
            embed[..., : (size[0] // self.patch_size), : (size[1] // self.patch_size)]
            for embed, size in zip(patch_embeds, image_sizes)
        ]

        patch_embeds = torch.cat([p.flatten(1).T for p in patch_embeds_list], dim=0).unsqueeze(0)
        patch_embeds = self.ln_pre(patch_embeds)

        # FIX: Ensure batch dimension exists for RoPE broadcast
        position_ids = position_ids_in_meshgrid(patch_embeds_list, self.config.image_size // self.config.patch_size)
        position_ids = position_ids.unsqueeze(0).to(patch_embeds.device)

        position_embeddings = self.patch_positional_embedding(patch_embeds, position_ids)
        attention_mask = generate_block_attention_mask([p.shape[-2] * p.shape[-1] for p in patch_embeds_list], patch_embeds)

        return self.transformer(patch_embeds, attention_mask, position_embeddings, output_hidden_states)

### Step 4: The Multimodal Projector
The projector takes the output of the vision tower and maps it into the text embedding space. It includes a `PatchMerger` which spatially downsamples the vision patches (e.g., 2x2 windows into 1) before applying a 2-layer MLP.

In [ ]:
class Mistral3RMSNormVision(nn.Module):
    def __init__(self, hidden_size: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(hidden_size))
        
    def forward(self, hidden_states: torch.Tensor) -> torch.Tensor:
        org_input_dtype = hidden_states.dtype
        hidden_states = hidden_states.to(torch.float32)
        variance = hidden_states.pow(2).mean(-1, keepdim=True)
        hidden_states = hidden_states * torch.rsqrt(variance + self.eps)
        return (self.weight * hidden_states).to(org_input_dtype)

class Mistral3PatchMerger(nn.Module):
    def __init__(self, config: Any) -> None:
        super().__init__()
        hidden_size = config.vision_config.hidden_size
        self.spatial_merge_size = config.spatial_merge_size
        self.patch_size = config.vision_config.patch_size
        self.merging_layer = nn.Linear(hidden_size * self.spatial_merge_size**2, hidden_size, bias=False)

    def forward(self, image_features: torch.Tensor, image_sizes: Iterable[torch.Tensor]) -> torch.Tensor:
        patch_grid_sizes = [(int(sz[0]) // self.patch_size, int(sz[1]) // self.patch_size) for sz in image_sizes]
        tokens_per_image = [h * w for h, w in patch_grid_sizes]
        embed_dim = image_features.shape[-1]
        permuted_tensor = []

        for image_index, image_tokens in enumerate(image_features.split(tokens_per_image)):
            h_patches, w_patches = patch_grid_sizes[image_index]
            image_grid = image_tokens.view(h_patches, w_patches, embed_dim).permute(2, 0, 1).unsqueeze(0)
            
            grid = torch.nn.functional.unfold(
                image_grid, kernel_size=self.spatial_merge_size, stride=self.spatial_merge_size
            )
            grid = grid.view(embed_dim * self.spatial_merge_size**2, -1).t()
            permuted_tensor.append(grid)

        merged_patches = torch.cat(permuted_tensor, dim=0)
        return self.merging_layer(merged_patches)

class Mistral3MultiModalProjector(nn.Module):
    def __init__(self, config: Any) -> None:
        super().__init__()
        self.norm = Mistral3RMSNormVision(config.vision_config.hidden_size, eps=config.text_config.rms_norm_eps)
        self.patch_merger = Mistral3PatchMerger(config)
        self.linear_1 = nn.Linear(config.vision_config.hidden_size, config.text_config.hidden_size, bias=False)
        self.act = nn.GELU()
        self.linear_2 = nn.Linear(config.text_config.hidden_size, config.text_config.hidden_size, bias=False)

    def forward(self, image_features: torch.Tensor, image_sizes: Iterable[torch.Tensor]) -> torch.Tensor:
        normed_patches = self.norm(image_features)
        merged_patches = self.patch_merger(normed_patches, image_sizes)
        return self.linear_2(self.act(self.linear_1(merged_patches)))

### Step 5: Text Backbone Configuration and Utilities
The text backbone uses Grouped Query Attention (GQA) and Llama-4 position-based attention scaling. 
*(Note: We applied the `field(default_factory=...)` fix here so the Python dataclass initializes safely without mutable default errors).*

In [ ]:
@dataclass
class RopeParameters:
    beta_fast: float = 32.0
    beta_slow: float = 1.0
    factor: float = 16.0
    llama_4_scaling_beta: float = 0.1
    mscale: float = 1.0
    mscale_all_dim: float = 1.0
    original_max_position_embeddings: int = 16384
    rope_theta: float = 1000000.0
    rope_type: str = "yarn"
    type: str = "yarn"

@dataclass
class Ministral3Config:
    attention_dropout: float = 0.0
    head_dim: int = 128
    hidden_size: int = 4096
    intermediate_size: int = 14336
    max_position_embeddings: int = 262144
    num_attention_heads: int = 32
    num_hidden_layers: int = 34
    num_key_value_heads: int = 8  
    rms_norm_eps: float = 1e-5
    
    # FIX: Use default_factory to prevent mutable default dataclass errors
    rope_parameters: dict = field(default_factory=lambda: RopeParameters().__dict__)
    vocab_size: int = 131072
    pad_token_id: Optional[int] = 11
    bos_token_id: Optional[int] = 1
    eos_token_id: Optional[int] = 2

def repeat_kv(hidden_states: torch.Tensor, n_rep: int) -> torch.Tensor:
    batch, num_kv_heads, seq_len, head_dim = hidden_states.shape
    if n_rep == 1: return hidden_states
    hidden_states = hidden_states[:, :, None, :, :].expand(batch, num_kv_heads, n_rep, seq_len, head_dim)
    return hidden_states.reshape(batch, num_kv_heads * n_rep, seq_len, head_dim)

def _get_llama_4_attn_scale(positions_ids: torch.Tensor, beta: float, max_pos: int) -> torch.Tensor:
    scaling = 1.0 + beta * torch.log(1.0 + torch.floor(positions_ids / max_pos))
    return scaling.unsqueeze(-1)

def create_causal_mask(config: Ministral3Config, inputs_embeds: torch.Tensor, attention_mask: Optional[torch.Tensor], past_key_values: Optional[KVCache] = None) -> Optional[torch.Tensor]:
    if inputs_embeds is None: return None
    batch_size, query_length, _ = inputs_embeds.shape
    device, dtype = inputs_embeds.device, inputs_embeds.dtype
    past_length = past_key_values.num_items() if past_key_values else 0
    kv_length = past_length + query_length
    
    neg_inf = torch.finfo(dtype).min
    mask = torch.full((query_length, kv_length), fill_value=neg_inf, device=device, dtype=dtype)
    mask = torch.triu(mask, diagonal=1 + past_length)

    sliding_window = getattr(config, "sliding_window", None)
    if sliding_window is not None:
        distance = (torch.arange(query_length, device=device) + past_length)[:, None] - torch.arange(kv_length, device=device)[None, :]
        mask = torch.where(distance <= sliding_window, mask, torch.full_like(mask, neg_inf))

    mask = mask.unsqueeze(0).unsqueeze(1).expand(batch_size, 1, query_length, kv_length)
    if attention_mask is not None:
        padding_mask = (1.0 - attention_mask).to(dtype) * neg_inf
        mask = mask + padding_mask[:, None, None, :]
    return mask

### Step 6: Text Decoder Architecture (Ministral Base)
Here we define the core Transformer blocks for the text model. 
*(Note: We implement `Ministral3Model` without an LM head to avoid the double-head bug when the multimodal wrapper wraps it).*

In [ ]:
class Ministral3RMSNorm(nn.Module):
    def __init__(self, hidden_size: int, eps: float = 1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(hidden_size))
        self.variance_epsilon = eps

    def forward(self, hidden_states: torch.Tensor) -> torch.Tensor:
        input_dtype = hidden_states.dtype
        x = hidden_states.to(torch.float32)
        x = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.variance_epsilon)
        return (self.weight * x).to(input_dtype)

class Ministral3RotaryEmbedding(nn.Module):
    def __init__(self, config: Ministral3Config):
        super().__init__()
        self.config = config
        dim = getattr(config, "head_dim", None) or (config.hidden_size // config.num_attention_heads)
        base = config.rope_parameters["rope_theta"]
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2, dtype=torch.float32) / float(dim)))
        self.register_buffer("inv_freq", inv_freq, persistent=False)

    def forward(self, x: torch.Tensor, position_ids: torch.LongTensor) -> Tuple[torch.Tensor, torch.Tensor]:
        batch = position_ids.shape[0]
        inv_freq_expanded = self.inv_freq[None, :, None].float().expand(batch, -1, 1).to(x.device)
        position_ids_expanded = position_ids[:, None, :].float()
        freqs = (inv_freq_expanded @ position_ids_expanded).transpose(1, 2)
        emb = torch.cat((freqs, freqs), dim=-1)
        return emb.cos().to(dtype=x.dtype), emb.sin().to(dtype=x.dtype)

class Ministral3Attention(nn.Module):
    def __init__(self, config: Ministral3Config, layer_idx: int):
        super().__init__()
        self.config, self.layer_idx = config, layer_idx
        self.head_dim = getattr(config, "head_dim", None) or (config.hidden_size // config.num_attention_heads)
        self.num_key_value_groups = config.num_attention_heads // config.num_key_value_heads
        self.scaling = self.head_dim ** -0.5

        self.q_proj = nn.Linear(config.hidden_size, config.num_attention_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(config.hidden_size, config.num_key_value_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(config.hidden_size, config.num_key_value_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(config.num_attention_heads * self.head_dim, config.hidden_size, bias=False)

    def forward(self, hidden_states, position_embeddings, attention_mask, cache_position, past_key_values):
        input_shape = hidden_states.shape[:-1]
        hidden_shape = (*input_shape, -1, self.head_dim)

        query_states = self.q_proj(hidden_states).view(hidden_shape).transpose(1, 2)
        key_states = self.k_proj(hidden_states).view(hidden_shape).transpose(1, 2)
        value_states = self.v_proj(hidden_states).view(hidden_shape).transpose(1, 2)

        cos, sin = position_embeddings
        query_states, key_states = apply_rotary_pos_emb(query_states, key_states, cos, sin)

        beta = self.config.rope_parameters.get("llama_4_scaling_beta", 0.1)
        max_pos = self.config.rope_parameters.get("original_max_position_embeddings", 16384)
        query_states = query_states * _get_llama_4_attn_scale(cache_position, beta, max_pos).to(query_states.dtype)

        if past_key_values is not None:
            key_states, value_states = past_key_values.update(key_states, value_states, self.layer_idx)

        k_states_rep = repeat_kv(key_states, self.num_key_value_groups)
        v_states_rep = repeat_kv(value_states, self.num_key_value_groups)

        attn_weights = torch.matmul(query_states, k_states_rep.transpose(2, 3)) * self.scaling
        if attention_mask is not None: attn_weights = attn_weights + attention_mask
        
        attn_weights = nn.functional.softmax(attn_weights, dim=-1, dtype=torch.float32).to(query_states.dtype)
        attn_output = torch.matmul(attn_weights, v_states_rep).transpose(1, 2).contiguous()
        return self.o_proj(attn_output.reshape(*input_shape, -1)), None

class Ministral3MLP(nn.Module):
    def __init__(self, config: Ministral3Config):
        super().__init__()
        self.gate_proj = nn.Linear(config.hidden_size, config.intermediate_size, bias=False)
        self.up_proj = nn.Linear(config.hidden_size, config.intermediate_size, bias=False)
        self.down_proj = nn.Linear(config.intermediate_size, config.hidden_size, bias=False)
        self.act_fn = nn.SiLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.down_proj(self.act_fn(self.gate_proj(x)) * self.up_proj(x))

class Ministral3DecoderLayer(nn.Module):
    def __init__(self, config: Ministral3Config, layer_idx: int):
        super().__init__()
        self.self_attn = Ministral3Attention(config=config, layer_idx=layer_idx)
        self.mlp = Ministral3MLP(config)
        self.input_layernorm = Ministral3RMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.post_attention_layernorm = Ministral3RMSNorm(config.hidden_size, eps=config.rms_norm_eps)

    def forward(self, hidden_states, attention_mask, cache_position, position_embeddings, past_key_values):
        residual = hidden_states
        hidden_states = self.input_layernorm(hidden_states)
        attn_out, _ = self.self_attn(hidden_states, position_embeddings, attention_mask, cache_position, past_key_values)
        hidden_states = residual + attn_out

        residual = hidden_states
        hidden_states = self.mlp(self.post_attention_layernorm(hidden_states))
        return residual + hidden_states

class Ministral3Model(nn.Module):
    """Base text model without the LM head."""
    def __init__(self, config: Ministral3Config):
        super().__init__()
        self.config = config
        self.embed_tokens = nn.Embedding(config.vocab_size, config.hidden_size, config.pad_token_id)
        self.layers = nn.ModuleList([Ministral3DecoderLayer(config, i) for i in range(config.num_hidden_layers)])
        self.norm = Ministral3RMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.rotary_emb = Ministral3RotaryEmbedding(config=config)

    def forward(self, inputs_embeds: torch.Tensor, attention_mask: Optional[torch.Tensor] = None, 
                cache_position: Optional[torch.Tensor] = None, past_key_values: Optional[KVCache] = None):
        
        if past_key_values is None:
            past_key_values = KVCache()

        if cache_position is None:
            past_len = past_key_values.num_items()
            cache_position = torch.arange(past_len, past_len + inputs_embeds.shape[1], device=inputs_embeds.device)

        position_ids = cache_position.unsqueeze(0)
        causal_mask = create_causal_mask(self.config, inputs_embeds, attention_mask, past_key_values)
        position_embeddings = self.rotary_emb(inputs_embeds, position_ids)

        hidden_states = inputs_embeds
        for layer in self.layers:
            hidden_states = layer(hidden_states, causal_mask, cache_position, position_embeddings, past_key_values)

        return {"last_hidden_state": self.norm(hidden_states), "past_key_values": past_key_values}

### Step 7: The Top-Level Multimodal Wrapper
Here we combine the Vision Tower, the Projector, and the Text Decoder. We implement a robust `_replace_image_tokens` function that finds the image placeholders in the text embeddings and seamlessly replaces them with the projected image features. Finally, we add the `lm_head` for causal language modeling.

In [ ]:
@dataclass
class Ministral3MultimodalConfig:
    spatial_merge_size: int = 2
    image_token_index: int = 10
    vision_feature_layer: int = -1
    tie_word_embeddings: bool = False
    text_config: Ministral3Config = field(default_factory=Ministral3Config)
    vision_config: PixtralConfig = field(default_factory=PixtralConfig)

class Ministral3MultimodalModel(nn.Module):
    def __init__(self, config: Ministral3MultimodalConfig):
        super().__init__()
        self.config = config

        # 1. Vision Encoder (Pixtral)
        self.vision_tower = PixtralVisionModel(config.vision_config)

        # 2. Projector
        self.multi_modal_projector = Mistral3MultiModalProjector(config)

        # 3. Text Encoder (Base model without LM head)
        self.language_model = Ministral3Model(config.text_config)

    def get_input_embeddings(self) -> nn.Embedding:
        return self.language_model.embed_tokens

    def _vision_forward_and_project(self, pixel_values: torch.Tensor, image_sizes: torch.Tensor) -> torch.Tensor:
        vision_outputs = self.vision_tower(pixel_values, image_sizes=image_sizes, output_hidden_states=True)

        if self.config.vision_feature_layer == -1:
            hv = vision_outputs["last_hidden_state"]
        else:
            hv = vision_outputs["hidden_states"][self.config.vision_feature_layer]

        if hv.dim() == 3 and hv.size(0) == 1:
            hv = hv.squeeze(0)

        return self.multi_modal_projector(hv, image_sizes)

    def _replace_image_tokens(self, input_ids: Optional[torch.Tensor], inputs_embeds: torch.Tensor, image_features: torch.Tensor) -> torch.Tensor:
        # Robust masking: works even if input_ids is None (useful for kv-cache decoding paradigms)
        if input_ids is None:
            image_token_tensor = to

### Step 8: Weight Loading and Inference
Finally, we write the logic to download weights from Hugging Face, stream them into our custom model architecture, and run a fast autoregressive generation loop using the KV cache.

In [ ]:
def load_weights_into_model(model, directory, device):
    """Streams .safetensors directly into the model object."""
    files = list(Path(directory).glob("*.safetensors"))
    if not files: raise FileNotFoundError(f"No safetensors found in {directory}")
    
    print(f"Loading {len(files)} weight files...")
    state_dict_keys = set(model.state_dict().keys())
    
    for file in files:
        with safe_open(file, framework="pt", device=device) as f:
            for key in f.keys():
                if key in state_dict_keys:
                    tensor = f.get_tensor(key).to(device=device, dtype=DTYPE)
                    # Helper to set nested params dynamically
                    module_name, param_name = key.rsplit(".", 1) if "." in key else ("", key)
                    submodule = model.get_submodule(module_name) if module_name else model
                    param = getattr(submodule, param_name)
                    
                    if param.shape != tensor.shape and param.numel() == tensor.numel():
                        tensor = tensor.view(param.shape)
                    with torch.no_grad():
                        param.data = tensor
                    del tensor
        gc.collect()
        torch.cuda.empty_cache()

@torch.no_grad()
def generate(model, processor, image, prompt, max_tokens=100, temp=0.7):
    # Format the prompt automatically using the processor's chat template
    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": prompt}]}]
    text_prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    
    inputs = processor(images=image, text=text_prompt, return_tensors="pt")
    input_ids = inputs["input_ids"].to(DEVICE)
    pixel_values = inputs["pixel_values"].to(DEVICE, dtype=DTYPE)
    image_sizes = inputs.get("image_sizes", None)
    if image_sizes is not None: image_sizes = image_sizes.to(DEVICE)

    generated_ids = []

    # --- Prefill Step (Process Image + Prompt) ---
    outputs = model(
        input_ids=input_ids,
        pixel_values=pixel_values,
        image_sizes=image_sizes,
        past_key_values=None 
    )
    
    next_logits = outputs["logits"][:, -1, :]
    kv_cache = outputs["past_key_values"]

    # --- Decode Loop (Autoregressive Generation) ---
    for _ in range(max_tokens):
        if temp > 0:
            probs = torch.softmax(next_logits / temp, dim=-1)
            next_token = torch.multinomial(probs, 1)
        else:
            next_token = torch.argmax(next_logits, dim=-1, keepdim=True)

        token_id = next_token.item()
        if token_id == processor.tokenizer.eos_token_id:
            break
            
        generated_ids.append(token_id)
        
        # Fast decode: Send ONLY the new token. Skip pixel_values!
        outputs = model(
            input_ids=next_token,
            pixel_values=None,   
            image_sizes=None,
            past_key_values=kv_cache
        )
        
        next_logits = outputs["logits"][:, -1, :]
        kv_cache = outputs["past_key_values"]

    return processor.tokenizer.decode(generated_ids, skip_special_tokens=True)

### Step 9: Running the Model!
Execute this cell to initialize the architecture, download the weights from `mistralai/Ministral-3-8B-Instruct-2512`, and generate text from an image.

In [ ]:
import gradio as gr

HF_REPO = "mistralai/Ministral-3-8B-Instruct-2512"
LOCAL_DIR = "./saved_model/Ministral3"

print("1. Downloading Weights and Processor...")
snapshot_download(
    repo_id=HF_REPO, 
    local_dir=LOCAL_DIR, 
    allow_patterns=["*.safetensors", "preprocessor_config.json", "tokenizer*", "chat_template.json"]
)
processor = AutoProcessor.from_pretrained(LOCAL_DIR)

print("2. Initializing Custom Architecture...")
config = Ministral3MultimodalConfig() # Defaults map to the 8B model!
model = Ministral3ForConditionalGeneration(config).to(DEVICE).to(DTYPE)

print("3. Loading Weights...")
load_weights_into_model(model, LOCAL_DIR, DEVICE)
model.eval()

print("4. Starting Gradio Interface...")

def run_inference(image, text, max_new, temp):
    if not image: 
        return "Please upload an image."
    text = text or "Describe this image in detail."
    
    try:
        # Call the generate function defined in Step 8
        return generate(model, processor, image, text, int(max_new), float(temp))
    except Exception as e:
        return f"Error: {str(e)}"

# Define the Gradio Layout
with gr.Blocks(title="Ministral-3 Zoo") as app:
    gr.Markdown(f"### Ministral-3 (8B) Multimodal on {DEVICE.upper()}")
    with gr.Row():
        img = gr.Image(type="pil", label="Image")
        with gr.Column():
            prompt = gr.Textbox(label="Prompt", value="Describe this image in detail.")
            tokens = gr.Slider(10, 1024, 256, label="Max Tokens")
            temp = gr.Slider(0.0, 1.5, 0.7, label="Temperature")
            btn = gr.Button("Generate", variant="primary")
            out = gr.Textbox(label="Output", lines=5)
    
    btn.click(run_inference, [img, prompt, tokens, temp], out)

# Launch the app inline in the Jupyter Notebook
app.launch(server_name="0.0.0.0", share=False, inline=True)